# H&M E-Commerce RecSys — End-to-End GPU Training & Evaluation Pipeline

Architecture: **6-Channel Native Retrieval (R1 Trend, R2 Repurchase, R3 Sibling, R4 ItemCF, R5 WALS, R6 Multi-Modal Two-Tower) → Candidate Union/RRF (Quota 270) → 49-Feature Point-in-Time Engineering → Single LightGBM LambdaMART Ranker → Validation & Final Test Evaluation**.

- **Hardware Profile**: Optimized for A100 80GB VRAM + 100GB+ Host RAM (`DuckDB memory_limit='64GB', threads=12`, `LightGBM n_jobs=12`, Two-Tower & Feature batch sizes `4096~8192`, CUDA AMP mixed precision, 100% GPU-vectorized internal validation via `torch.matmul` + `torch.topk`, and explicit VRAM cleanup).
- **Two Execution Modes**: Set `MODE = "full"` (or `"smoke"`) in **Cell 3** and click **Run All**.

## Cell 1: Hardware & Environment Probe
Verify that a GPU accelerator (CUDA / MPS) is online, and inspect the GPU model, total/free VRAM, CPU logical cores, and host physical RAM.

In [ ]:
import os
import platform
import shutil
import psutil
import torch

if torch.cuda.is_available():
    ACCELERATOR = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    vram_info = f"{total_vram / (1024**3):.2f} GiB total ({free_vram / (1024**3):.2f} GiB free)"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    ACCELERATOR = "mps"
    gpu_name = "Apple Silicon MPS GPU"
    vram_info = "Unified Memory"
else:
    raise RuntimeError(
        "GPU environment required: Neither CUDA nor MPS accelerator is online. "
        "Please attach a GPU runtime (e.g., Colab A100 GPU) before running."
    )

vm = psutil.virtual_memory()
disk = shutil.disk_usage(".")
print("=" * 72)
print(f"[Hardware Probe] Accelerator : {ACCELERATOR.upper()} ({gpu_name})")
print(f"[Hardware Probe] GPU VRAM    : {vram_info}")
print(f"[Hardware Probe] PyTorch     : {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"[Hardware Probe] CPU Cores   : {os.cpu_count()} logical cores ({platform.machine()})")
print(f"[Hardware Probe] Host RAM    : {vm.available / (1024**3):.2f} GiB available / {vm.total / (1024**3):.2f} GiB total")
print(f"[Hardware Probe] Disk Free   : {disk.free / (1024**3):.2f} GiB free")
print("=" * 72)

## Cell 2: Workspace Initialization & Dependency Check
Automatically locate the repository root (mounting Google Drive on Colab if needed), configure `sys.path`, and verify all required compute libraries.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

IS_COLAB = sys.platform == "linux" and bool(
    os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_BACKEND_VERSION")
)

if IS_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/e_commerce_recsys")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys")
    if not (PROJECT_ROOT / "src/hm_recsys").exists():
        if (DRIVE_PROJECT_ROOT / "src/hm_recsys").exists():
            PROJECT_ROOT = DRIVE_PROJECT_ROOT
        else:
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth=1",
                    "https://github.com/HaokunLiu-Vincent/ecommerce-recsys.git",
                    str(PROJECT_ROOT),
                ],
                check=True,
            )
    SOURCE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data"
    LOCAL_DATA_ROOT = Path("/content/hm_local_data")
    PERSISTENT_ROOT = SOURCE_DATA_ROOT / "runs/colab"
    MOUNT_ROOT = Path("/content/drive")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "src/hm_recsys").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / "src/hm_recsys").exists():
        raise FileNotFoundError("Could not locate project root containing src/hm_recsys")
    SOURCE_DATA_ROOT = PROJECT_ROOT / "data"
    LOCAL_DATA_ROOT = PROJECT_ROOT / "data"
    PERSISTENT_ROOT = PROJECT_ROOT / "data/runs/persistent"
    MOUNT_ROOT = None

SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
os.environ["PYTHONPATH"] = SRC_PATH

REQUIRED_PACKAGES = (
    "numpy",
    "pyarrow",
    "duckdb",
    "scipy",
    "sklearn",
    "implicit",
    "torch",
    "faiss",
    "lightgbm",
    "psutil",
)
missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
if missing:
    raise RuntimeError(f"Missing required dependencies: {missing}")

print(f"[Workspace] Project Root    : {PROJECT_ROOT}")
print(f"[Workspace] Source Data     : {SOURCE_DATA_ROOT}")
print(f"[Workspace] Local Work Data : {LOCAL_DATA_ROOT}")
print(f"[Workspace] Persistent Runs : {PERSISTENT_ROOT}")
print("[Workspace] All core dependencies verified.")

## Cell 3: Run Configuration Declaration
Select `MODE = "full"` (production run on all cutoffs with automatic Final Test evaluation) or `MODE = "smoke"` (fast connectivity and contract verification with `eval_test = False`). All downstream batch sizes, thread counts, and stage gates are derived automatically.

In [ ]:
from datetime import datetime
from hm_recsys.runtime.modular import resolved_profile

MODE = "full"  # @param ["full", "smoke"]

if MODE not in ("full", "smoke"):
    raise ValueError(f"Invalid MODE={MODE!r}; must be 'full' or 'smoke'")

EVAL_TEST = MODE == "full"
PIPELINE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROFILE = resolved_profile(
    PROJECT_ROOT,
    MODE,
    device=PIPELINE_DEVICE,
    performance="optimized",
)

# Unbind hardware resources for high-memory GPU execution (A100 80GB + 100GB RAM)
if MODE == "full":
    PROFILE["feature_query_batch"] = 4096
    PROFILE["compute"]["tt_batch_size"] = 4096
    PROFILE["compute"]["threads"] = 12
    PROFILE["compute"]["loader_workers"] = 4
    PROFILE["compute"]["interop_threads"] = 4

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"hm_{MODE}_{TIMESTAMP}"
LOCAL_RUN_DIR = LOCAL_DATA_ROOT / "runs/colab" / RUN_ID
PERSISTENT_RUN_DIR = PERSISTENT_ROOT / RUN_ID

print("=" * 72)
print(f"[Config] Mode                : {MODE}")
print(f"[Config] Run ID              : {RUN_ID}")
print(f"[Config] Eval Final Test     : {EVAL_TEST}")
print(f"[Config] Pipeline Device     : {PROFILE['device']} (Accelerator: {ACCELERATOR})")
print(f"[Config] DuckDB Config       : memory_limit='64GB', threads=12")
print(f"[Config] LightGBM Threads    : n_jobs=12")
print(f"[Config] Two-Tower Batch     : {PROFILE['compute']['tt_batch_size']} (Epochs: {PROFILE['compute']['tt_epochs']})")
print(f"[Config] Feature Query Batch : {PROFILE['feature_query_batch']}")
print("=" * 72)

## Cell 4: End-to-End Main Pipeline Execution
Single entry point executing input staging, data preparation, 6-channel recall (`trend`, `repurchase`, `sibling`, `itemcf`, `wals`, `twotower` with pure-GPU internal validation), candidate fusion, 49-dimensional feature engineering, LightGBM LambdaMART training, Validation, fresh-process Reload, and (when `MODE == "full"`) Final Test evaluation.

In [ ]:
import gc
import time
import traceback
from hm_recsys.core.contracts import read_json
from hm_recsys.runtime.files import stage_inputs
from hm_recsys.runtime.modular import ORDER, Session


def clear_vram_and_ram(tag: str = "") -> None:
    """Force garbage collection and empty CUDA cache between pipeline stages."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        alloc_mb = torch.cuda.memory_allocated(0) / (1024**2)
        res_mb = torch.cuda.memory_reserved(0) / (1024**2)
        if tag:
            print(f"  [Memory Cleanup @ {tag}] CUDA Allocated: {alloc_mb:.1f} MiB | Reserved: {res_mb:.1f} MiB")


STAGE_TIMINGS: dict[str, float] = {}
STAGE_REPORTS: dict[str, dict] = {}
PIPELINE_START = time.perf_counter()

try:
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

    if SOURCE_DATA_ROOT.resolve() != LOCAL_DATA_ROOT.resolve():
        print("[Stage 0/13] Staging immutable source inputs to local storage...")
        t0 = time.perf_counter()
        staging_info = stage_inputs(SOURCE_DATA_ROOT, LOCAL_DATA_ROOT)
        STAGE_TIMINGS["stage_inputs"] = time.perf_counter() - t0
        print(f"  -> Staged inputs verified in {STAGE_TIMINGS['stage_inputs']:.1f}s (digest: {staging_info['input_digest'][:16]}...)")

    with Session(
        PROJECT_ROOT,
        LOCAL_DATA_ROOT,
        LOCAL_RUN_DIR,
        PERSISTENT_RUN_DIR,
        profile=PROFILE,
        mount_root=MOUNT_ROOT,
        recover_lock=True,
    ) as session:
        total_stages = len(ORDER) + (1 if EVAL_TEST else 0)
        for idx, stage_name in enumerate(ORDER, start=1):
            print(f"\n[Stage {idx}/{total_stages}] Running '{stage_name}'...")
            t0 = time.perf_counter()
            report = session.run(stage_name, action="run")
            elapsed = time.perf_counter() - t0
            STAGE_TIMINGS[stage_name] = elapsed
            STAGE_REPORTS[stage_name] = report
            print(f"  -> Completed '{stage_name}' in {elapsed:.2f}s")
            if stage_name in ("wals", "twotower", "features", "ranker", "reload"):
                clear_vram_and_ram(stage_name)

        if EVAL_TEST:
            print(f"\n[Stage {total_stages}/{total_stages}] Running 'final_test' (authorized via frozen Validation lock)...")
            t0 = time.perf_counter()
            lock_data = read_json(LOCAL_RUN_DIR / "validation_lock.json")
            test_report = session.run(
                "final_test",
                action="run",
                run_final_test=True,
                approve_final_test=True,
                locked_signature=lock_data["signature"],
            )
            elapsed = time.perf_counter() - t0
            STAGE_TIMINGS["final_test"] = elapsed
            STAGE_REPORTS["final_test"] = test_report
            print(f"  -> Completed 'final_test' in {elapsed:.2f}s")
            clear_vram_and_ram("final_test")

    TOTAL_PIPELINE_SECONDS = time.perf_counter() - PIPELINE_START
    print("\n" + "=" * 72)
    print(f"[SUCCESS] End-to-End Pipeline ({MODE.upper()}) finished in {TOTAL_PIPELINE_SECONDS:.2f}s")
    print("=" * 72)
except Exception as exc:
    print("\n" + "!" * 72)
    print(f"[PIPELINE ERROR] Execution failed during MODE={MODE!r}: {exc}")
    print("!" * 72)
    traceback.print_exc()
    raise
finally:
    clear_vram_and_ram("pipeline_exit")

## Cell 5: Core Metrics & Artifacts Dashboard
Summarize per-stage execution times, Validation `MAP@12`, `Recall@12`, `NDCG@12`, per-channel Recall@K, and (when `MODE == "full"`) Final Test offline evaluation metrics.

In [ ]:
from hm_recsys.core.contracts import read_json

val_report_path = LOCAL_RUN_DIR / "reports/validation.json"
test_report_path = LOCAL_RUN_DIR / "reports/final_test.json"

print("=" * 72)
print(f"H&M RECSYS EXECUTION DASHBOARD — RUN_ID: {RUN_ID} (MODE: {MODE.upper()})")
print("=" * 72)

print("\n1. Stage Timing Breakdown:")
print("-" * 44)
for stage_key, secs in STAGE_TIMINGS.items():
    print(f"  {stage_key:<20s} : {secs:>8.2f} s ({secs / 60.0:>6.2f} min)")
print("-" * 44)
total_recorded = sum(STAGE_TIMINGS.values())
print(f"  {'TOTAL':<20s} : {total_recorded:>8.2f} s ({total_recorded / 60.0:>6.2f} min)")

if val_report_path.is_file():
    val_report = read_json(val_report_path)
    val_metrics = val_report.get("native_metrics", {})
    rrf_metrics = val_report.get("baselines", {})
    print("\n2. Global Validation Metrics (Cutoff: 2020-09-08):")
    print("-" * 44)
    print(f"  LambdaMART MAP@12    : {val_metrics.get('map12', 0.0):.6f}")
    print(f"  LambdaMART Recall@12 : {val_metrics.get('recall12', 0.0):.6f}")
    print(f"  LambdaMART NDCG@12   : {val_metrics.get('ndcg12', 0.0):.6f}")
    print(f"  Native Union Recall  : {val_metrics.get('candidate_recall', 0.0):.6f}")
    print(f"  RRF Baseline MAP@12  : {rrf_metrics.get('map12', 0.0):.6f}")
    print(f"  Eligible Queries     : {val_metrics.get('eligible_queries', 'N/A')}")

    retrieval_info = val_report.get("retrieval", {})
    if retrieval_info:
        print("\n3. Validation Per-Channel Native Recall:")
        print("-" * 44)
        for ch, ch_data in retrieval_info.items():
            k_quota = ch_data.get("quota", "K")
            rec_val = ch_data.get("recall", 0.0)
            print(f"  {ch:<12s} (Top-{str(k_quota):<2s}) : Recall = {rec_val:.6f} | Rows = {ch_data.get('rows', 0)}")

if EVAL_TEST and test_report_path.is_file():
    test_report = read_json(test_report_path)
    test_metrics = test_report.get("test_metrics", test_report)
    print("\n4. Final Test Metrics (Cutoff: 2020-09-15):")
    print("-" * 44)
    print(f"  Final Test MAP@12    : {test_metrics.get('map12', 0.0):.6f}")
    print(f"  Final Test Recall@12 : {test_metrics.get('recall12', 0.0):.6f}")
    print(f"  Final Test NDCG@12   : {test_metrics.get('ndcg12', 0.0):.6f}")
    print(f"  Final Test HitRate12 : {test_metrics.get('hit_rate12', 0.0):.6f}")
    print(f"  Candidate Recall     : {test_metrics.get('candidate_recall', 0.0):.6f}")
    print(f"  Eligible Queries     : {test_metrics.get('eligible_queries', 'N/A')}")
elif not EVAL_TEST:
    print("\n4. Final Test Metrics: SKIPPED (eval_test=False in 'smoke' mode)")

print("\n5. Persisted Artifact Paths:")
print("-" * 44)
print(f"  Local Run Directory      : {LOCAL_RUN_DIR}")
print(f"  Persistent Run Directory : {PERSISTENT_RUN_DIR}")
print("=" * 72)